In [8]:
%pip install google-api-python-client google-auth-oauthlib google-auth-httplib2

Note: you may need to restart the kernel to use updated packages.


In [9]:
import os
import pickle as pkl
from googleapiclient.discovery import build
from google_auth_oauthlib.flow import InstalledAppFlow
from google.auth.transport.requests import Request

SCOPES = ['https://www.googleapis.com/auth/youtube.force-ssl']

def setup_youtube_oauth():
    creds = None
    token_path = '../token.pickle'

    # Cek token lama yang masih valid
    if os.path.exists(token_path):
        with open(token_path, 'rb') as token:
            creds = pkl.load(token)

    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            # Refresh token otomatis
            creds.refresh(Request())
            print('Token berhasil di-refresh!')
        else:
            # Login baru: buka browser otomatis, tidak perlu copy-paste kode
            flow = InstalledAppFlow.from_client_secrets_file(
                '../client_secrets.json',
                scopes=SCOPES
            )
            print('Membuka browser untuk login YouTube...')
            print('Jika browser tidak terbuka otomatis, buka URL yang muncul di terminal.')
            # run_local_server: buka browser & handle callback otomatis di localhost
            creds = flow.run_local_server(
                port=0,           # port otomatis (bebas konflik)
                open_browser=True
            )

        # Simpan token untuk sesi berikutnya
        with open(token_path, 'wb') as token:
            pkl.dump(creds, token)
        print('Token disimpan!')

    youtube = build('youtube', 'v3', credentials=creds)
    print('Berhasil login ke YouTube!')
    return youtube

youtube = setup_youtube_oauth()


Berhasil login ke YouTube!


In [10]:
import pickle

# Load model Random Forest
model_path = '../model/model_baseline.pkl'
tfidf_path = '../model/tfidf_vectorizer.pkl'

with open(model_path, 'rb') as f:
    model = pickle.load(f)

with open(tfidf_path, 'rb') as f:
    tfidf = pickle.load(f)

print("✅ Model berhasil dimuat!")
print("🤖 Siap mendeteksi komentar judi...")

✅ Model berhasil dimuat!
🤖 Siap mendeteksi komentar judi...


In [11]:
import re
import unicodedata

# ============================================
# BAGIAN 1: KONVERSI UNICODE FONT
# ============================================
UNICODE_FONT_MAP = {}
unicode_ranges = [
    (0x1D400, 'A'), (0x1D41A, 'a'),
    (0x1D434, 'A'), (0x1D44E, 'a'),
    (0x1D468, 'A'), (0x1D482, 'a'),
    (0x1D49C, 'A'), (0x1D4B6, 'a'),
    (0x1D4D0, 'A'), (0x1D4EA, 'a'),
    (0x1D504, 'A'), (0x1D51E, 'a'),
    (0x1D538, 'A'), (0x1D552, 'a'),
    (0x1D56C, 'A'), (0x1D586, 'a'),
    (0x1D5A0, 'A'), (0x1D5BA, 'a'),
    (0x1D5D4, 'A'), (0x1D5EE, 'a'),
    (0x1D608, 'A'), (0x1D622, 'a'),
    (0x1D63C, 'A'), (0x1D656, 'a'),
    (0x1D670, 'A'), (0x1D68A, 'a'),
]
for start, base_char in unicode_ranges:
    for i in range(26):
        UNICODE_FONT_MAP[chr(start + i)] = chr(ord(base_char) + i)
for i in range(10):
    UNICODE_FONT_MAP[chr(0x1D7CE + i)] = str(i)

# ============================================
# BAGIAN 2: SIMBOL PENGGANTI HURUF
# ============================================
SIMBOL_MAP = {'@': 'a', '|': 'l', '$': 's', '!': 'i'}

# ============================================
# BAGIAN 3: KEYWORD YANG PASTI JUDI
# (meskipun berdiri sendiri)
# ============================================
KEYWORD_PASTI_JUDI_TUNGGAL = [
    # Istilah utama judi online
    'gacor', 'maxwin', 'scatter', 'jackpot', 'togel',
    'withdraw', 'rungkad', 'rungkat', 'freebet',
    'wa admin', 'hub admin', 'link di bio', 'klik link',
    'daftar sekarang', 'join sekarang', 'menang terus',
    'situs gacor', 'situs terpercaya', 'agen resmi',
    'garansi kekalahan', 'anti rungkad', 'pasti bayar',
    'bocoran', 'pola gacor', 'rtp live', 'link rtp',
    'jam gacor', 'trik gacor', 'info pusat',
    'new member', 'bonus new member', 'member baru',
    'anti nawala', 'link alternatif',

    # Nama game judi spesifik
    'mahjong ways', 'sweet bonanza', 'sugar rush',
    'starlight princess', 'spaceman', 'kakek zeus',
    'kakek merah', 'gatot kaca', 'gatotkaca',

    # Provider judi
    'pragmatic', 'pg soft', 'joker123', 'joker gaming',
    'spadegaming', 'habanero', 'microgaming',
    'sbobet', 'pkv',

    # Jenis judi lain
    'parlay', 'mix parlay', 'roulette', 'baccarat',
    'sicbo', 'bandarq', 'judol',

    # Metode transaksi judi
    'depo pulsa', 'tanpa potongan', 'auto wd',
    'jamin wd', 'anti sedot', 'pasti bayar',

    # Leetspeak / bypass filter
    'sl0t', 's l o t', 's|ot', '5lot',
    'gac0r', 'g4cor', 'z3us', 'zeu5',
    'sketer', 'jekpot', 'm4xwin', 'maxw1n',
    'g4c0r', '5l0t', 'sl0t gacor',
]

# ============================================
# BAGIAN 4: KEYWORD YANG BUTUH KOMBINASI
# (1 kata saja tidak cukup, harus ada 2+)
# ============================================
KEYWORD_KOMBINASI_JUDI = [
    'slot', 'casino', 'poker', 'bet', 'spin',
    'bonus', 'deposit', 'jp', 'cuan', 'modal',
    'daftar', 'join', 'profit', 'menang',
    'hoki', 'withdraw', 'depo', 'wd', 'wede',
    'zeus', 'olympus', 'mahjong', 'princess',
    'x500', 'x250', 'pecah', 'rtp', 'pola',
    'to', 'turnover', 'taruhan', 'judi',
    'vip', 'slotter', 'sikat', 'cek profil',
    'qris', 'dana', 'ovo', 'gopay',
]

# ============================================
# BAGIAN 5: FRASA KOMBINASI YANG PASTI JUDI
# ============================================
FRASA_JUDI = [
    r'slot\s+gacor', r'slot\s+online', r'slot\s+\d+',
    r'daftar\s+slot', r'main\s+slot', r'main\s+casino',
    r'bonus\s+deposit', r'bonus\s+new\s+member',
    r'bonus\s+slot', r'bonus\s+cashback',
    r'jp\s+terus', r'jp\s+maxwin',
    r'link\s+di\s+bio', r'klik\s+bio', r'cek\s+bio',
    r'modal\s+\d+\s*(rb|ribu|k|jt|juta)',
    r'cuan\s+\d+\s*(rb|ribu|k|jt|juta)',
    r'menang\s+\d+\s*(jt|juta|rb|ribu)',
    r'wd\s+\d+\s*(jt|juta|rb|ribu)',
    r'depo\s+\d+\s*(rb|ribu|k)',
    r'x\s*\d+\s*(lipat|kali)',
    r'via\s+(dana|ovo|gopay|qris)\s+(slot|casino|deposit)',
    r'(slot|casino|poker)\s+(via|pakai|pake)\s+\w+',
]

# ============================================
# BAGIAN 6: POLA NAMA SITUS JUDI
# ============================================
POLA_NAMA_SITUS = [
    r'raja\s*\d+', r'pulau\s*\d+', r'slot\s*\d+',
    r'spin\s*\d+', r'jp\s*\d+', r'maxwin\s*\d+',
    r'hoki\s*\d+', r'gacor\s*\d+', r'bet\s*\d+',
    r'\w+\s*(777|303|88|99|168|365|4d|2d)\b',
]

# ============================================
# BAGIAN 7: FUNGSI NORMALISASI TEKS
# ============================================
def konversi_unicode_font(teks):
    hasil = ''
    for char in str(teks):
        if char in UNICODE_FONT_MAP:
            hasil += UNICODE_FONT_MAP[char]
        else:
            hasil += unicodedata.normalize('NFKC', char)
    return hasil

def ganti_simbol_di_kata(teks):
    """Ganti angka pengganti huruf HANYA di dalam kata"""
    teks = re.sub(r'(?<=[a-z])4(?=[a-z])', 'a', teks)
    teks = re.sub(r'(?<=[a-z])3(?=[a-z])', 'e', teks)
    teks = re.sub(r'(?<=[a-z])0(?=[a-z])', 'o', teks)
    teks = re.sub(r'(?<=[a-z])1(?=[a-z])', 'i', teks)
    teks = re.sub(r'(?<=[a-z])5(?=[a-z])', 's', teks)
    return teks

def normalisasi_teks(teks):
    """Pipeline normalisasi lengkap"""
    teks = str(teks)
    teks = konversi_unicode_font(teks)   # unicode bold/italic → normal
    teks = teks.lower()                   # lowercase
    for simbol, huruf in SIMBOL_MAP.items():
        teks = teks.replace(simbol, huruf) # @ → a, | → l, dll
    teks = ganti_simbol_di_kata(teks)     # g4cor → gacor
    teks = re.sub(r'http\S+|www\S+|bit\.ly\S+', '', teks)  # hapus URL
    teks = re.sub(r'([a-z])(\d)', r'\1 \2', teks)  # pisah huruf+angka
    teks = re.sub(r'(\d)([a-z])', r'\1 \2', teks)
    teks = re.sub(r'[^\w\s]', ' ', teks)  # hapus karakter aneh
    teks = re.sub(r'\s+', ' ', teks).strip()
    return teks

# ============================================
# BAGIAN 8: FUNGSI DETEKSI UTAMA
# ============================================
def adalah_judi(teks_komentar):
    """
    5 lapis deteksi:
    1. Keyword tunggal pasti judi
    2. Frasa kombinasi pasti judi
    3. Pola nama situs
    4. Minimal 2 keyword kombinasi
    5. Model ML threshold 70%
    """
    teks_bersih = normalisasi_teks(teks_komentar)

    # Lapis 1: Keyword tunggal pasti judi
    for kw in KEYWORD_PASTI_JUDI_TUNGGAL:
        if kw in teks_bersih:
            return True, f"keyword pasti: '{kw}'"

    # Lapis 2: Frasa kombinasi pasti judi
    for frasa in FRASA_JUDI:
        if re.search(frasa, teks_bersih, re.IGNORECASE):
            return True, f"frasa judi: '{frasa}'"

    # Lapis 3: Pola nama situs
    for pola in POLA_NAMA_SITUS:
        if re.search(pola, teks_bersih, re.IGNORECASE):
            return True, f"pola situs: '{pola}'"

    # Lapis 4: Minimal 2 keyword kombinasi
    keyword_ditemukan = [
        kw for kw in KEYWORD_KOMBINASI_JUDI
        if kw in teks_bersih
    ]
    if len(keyword_ditemukan) >= 2:
        return True, f"kombinasi: {keyword_ditemukan}"

    # Lapis 5: Model ML threshold 70%
    teks_vector = tfidf.transform([teks_bersih])
    confidence = model.predict_proba(teks_vector)[0]
    pct_judi = confidence[1] * 100

    if pct_judi >= 70:
        return True, f"model ML: {pct_judi:.1f}%"

    return False, f"normal ({pct_judi:.1f}% judi)"

# ============================================
# TEST SEMUA KASUS
# ============================================
test_komentar = [
    # Seharusnya NORMAL
    ("mantap bang, kira2 bonus dari video ini dapat berapa?", "NORMAL"),
    ("wah videonya bagus banget kak sukses terus",            "NORMAL"),
    ("kapan upload lagi bang?",                               "NORMAL"),
    ("keren banget editannya, pakai software apa?",           "NORMAL"),
    ("via ovo bisa transfer ke bank lain ga?",                "NORMAL"),

    # Seharusnya JUDI
    ("daftar sekarang slot gacor maxwin jp terus",            "JUDI"),
    ("bonus deposit 100% new member buruan join",             "JUDI"),
    ("RAJA303 terbaik hari ini gaskeun",                      "JUDI"),
    ("mau cuan? hub admin wa di profil bio",                  "JUDI"),
    ("main slot modal 50rb balik 2jt",                        "JUDI"),
    ("𝒔𝒍𝒐𝒕 𝒈𝒂𝒄𝒐𝒓 hari ini buruan daftar",               "JUDI"),
    ("PULAu777 lagi hoki banget hari ini",                    "JUDI"),
    ("sweet bonanza pecah terus hari ini gaskeun",            "JUDI"),
]

print("=" * 65)
print("🧪 TEST DETEKSI 5 LAPIS")
print("=" * 65)

benar = 0
for teks, ekspektasi in test_komentar:
    hasil, alasan = adalah_judi(teks)
    prediksi = "JUDI" if hasil else "NORMAL"
    status = "✅" if prediksi == ekspektasi else "❌ SALAH"
    if prediksi == ekspektasi:
        benar += 1
    print(f"{status} [{ekspektasi}→{prediksi}]")
    print(f"   Teks   : {teks[:60]}")
    print(f"   Alasan : {alasan}")
    print()

print(f"Akurasi test: {benar}/{len(test_komentar)} "
      f"({benar/len(test_komentar)*100:.0f}%)")
print("\n✅ Cell 4 selesai! Lanjut ke Cell 5.")

🧪 TEST DETEKSI 5 LAPIS
✅ [NORMAL→NORMAL]
   Teks   : mantap bang, kira2 bonus dari video ini dapat berapa?
   Alasan : normal (39.0% judi)

✅ [NORMAL→NORMAL]
   Teks   : wah videonya bagus banget kak sukses terus
   Alasan : normal (54.0% judi)

✅ [NORMAL→NORMAL]
   Teks   : kapan upload lagi bang?
   Alasan : normal (6.0% judi)

✅ [NORMAL→NORMAL]
   Teks   : keren banget editannya, pakai software apa?
   Alasan : normal (35.0% judi)

✅ [NORMAL→NORMAL]
   Teks   : via ovo bisa transfer ke bank lain ga?
   Alasan : normal (6.0% judi)

✅ [JUDI→JUDI]
   Teks   : daftar sekarang slot gacor maxwin jp terus
   Alasan : keyword pasti: 'gacor'

✅ [JUDI→JUDI]
   Teks   : bonus deposit 100% new member buruan join
   Alasan : keyword pasti: 'new member'

✅ [JUDI→JUDI]
   Teks   : RAJA303 terbaik hari ini gaskeun
   Alasan : pola situs: 'raja\s*\d+'

✅ [JUDI→JUDI]
   Teks   : mau cuan? hub admin wa di profil bio
   Alasan : keyword pasti: 'hub admin'

✅ [JUDI→JUDI]
   Teks   : main slot modal 50rb

In [12]:
def adalah_judi(teks_komentar):
    """
    5 lapis deteksi:
    1. Keyword tunggal pasti judi
    2. Frasa kombinasi pasti judi
    3. Pola nama situs
    4. Minimal 2 keyword kombinasi
    5. Model ML threshold 70%
    """
    teks_bersih = normalisasi_teks(teks_komentar)

    # Lapis 1: Keyword tunggal pasti judi
    for kw in KEYWORD_PASTI_JUDI_TUNGGAL:
        if kw in teks_bersih:
            return True, f"keyword pasti: '{kw}'"

    # Lapis 2: Frasa kombinasi pasti judi
    for frasa in FRASA_JUDI:
        if re.search(frasa, teks_bersih, re.IGNORECASE):
            return True, f"frasa judi: '{frasa}'"

    # Lapis 3: Pola nama situs
    for pola in POLA_NAMA_SITUS:
        if re.search(pola, teks_bersih, re.IGNORECASE):
            return True, f"pola situs: '{pola}'"

    # Lapis 4: Minimal 2 keyword kombinasi
    keyword_ditemukan = [
        kw for kw in KEYWORD_KOMBINASI_JUDI
        if kw in teks_bersih
    ]
    if len(keyword_ditemukan) >= 2:
        return True, f"kombinasi: {keyword_ditemukan}"

    # Lapis 5: Model ML threshold 70%
    teks_vector = tfidf.transform([teks_bersih])
    confidence = model.predict_proba(teks_vector)[0]
    pct_judi = confidence[1] * 100

    if pct_judi >= 70:
        return True, f"model ML: {pct_judi:.1f}%"

    return False, f"normal ({pct_judi:.1f}% judi)"

# ============================================
# Test fungsi
# ============================================
test_komentar = [
    # Seharusnya NORMAL
    ("mantap bang, kira2 bonus dari video ini dapat berapa?", "NORMAL"),
    ("wah videonya bagus banget kak",                         "NORMAL"),
    ("kapan upload lagi bang?",                               "NORMAL"),
    ("via ovo bisa transfer ke bank lain ga?",                "NORMAL"),

    # Seharusnya JUDI
    ("RAJA303 terbaik hari ini",                              "JUDI"),
    ("wah videonya bagus banget",                             "NORMAL"),
    ("daftar slot gacor sekarang",                            "JUDI"),
    ("mantap bang kontennya",                                 "NORMAL"),
    ("bonus deposit 100% new member buruan",                  "JUDI"),
    ("mau cuan? hub admin wa di profil bio",                  "JUDI"),
    ("sweet bonanza pecah terus hari ini",                    "JUDI"),
    ("PULAu777 lagi hoki banget",                             "JUDI"),
]

print("=" * 65)
print("🧪 TEST DETEKSI 5 LAPIS")
print("=" * 65)

benar = 0
for teks, ekspektasi in test_komentar:
    hasil, alasan = adalah_judi(teks)
    prediksi = "JUDI" if hasil else "NORMAL"
    status = "✅" if prediksi == ekspektasi else "❌ SALAH"
    if prediksi == ekspektasi:
        benar += 1
    print(f"{status} [{ekspektasi}→{prediksi}]")
    print(f"   Teks   : {teks}")
    print(f"   Alasan : {alasan}")
    print()

print(f"Akurasi test: {benar}/{len(test_komentar)} "
      f"({benar/len(test_komentar)*100:.0f}%)")
print("\n✅ Cell 5 selesai! Lanjut ke Cell 6A.")

🧪 TEST DETEKSI 5 LAPIS
✅ [NORMAL→NORMAL]
   Teks   : mantap bang, kira2 bonus dari video ini dapat berapa?
   Alasan : normal (39.0% judi)

✅ [NORMAL→NORMAL]
   Teks   : wah videonya bagus banget kak
   Alasan : normal (14.0% judi)

✅ [NORMAL→NORMAL]
   Teks   : kapan upload lagi bang?
   Alasan : normal (6.0% judi)

✅ [NORMAL→NORMAL]
   Teks   : via ovo bisa transfer ke bank lain ga?
   Alasan : normal (6.0% judi)

✅ [JUDI→JUDI]
   Teks   : RAJA303 terbaik hari ini
   Alasan : pola situs: 'raja\s*\d+'

✅ [NORMAL→NORMAL]
   Teks   : wah videonya bagus banget
   Alasan : normal (20.0% judi)

✅ [JUDI→JUDI]
   Teks   : daftar slot gacor sekarang
   Alasan : keyword pasti: 'gacor'

✅ [NORMAL→NORMAL]
   Teks   : mantap bang kontennya
   Alasan : normal (11.0% judi)

✅ [JUDI→JUDI]
   Teks   : bonus deposit 100% new member buruan
   Alasan : keyword pasti: 'new member'

✅ [JUDI→JUDI]
   Teks   : mau cuan? hub admin wa di profil bio
   Alasan : keyword pasti: 'hub admin'

✅ [JUDI→JUDI]
   Teks

In [13]:
# Verifikasi client_secrets.json tersedia secara lokal
import os, shutil

# File client_secrets.json ada di root folder judol-detector/
client_secret_src = '../client_secrets.json'
client_secret_local = 'client_secrets.json'

if os.path.exists(client_secret_src):
    # Salin ke folder notebooks/ supaya mudah diakses
    shutil.copy(client_secret_src, client_secret_local)
    print('client_secrets.json ditemukan dan siap digunakan!')
elif os.path.exists(client_secret_local):
    print('client_secrets.json ditemukan di folder notebooks!')
else:
    raise FileNotFoundError(
        'client_secrets.json tidak ditemukan!\n'
        'Pastikan file ada di: d:/Detect judol/judol-detector/client_secrets.json'
    )

print('Siap melanjutkan ke autentikasi YouTube API!')


client_secrets.json ditemukan dan siap digunakan!
Siap melanjutkan ke autentikasi YouTube API!


In [14]:
def ambil_komentar_terbaru(youtube, video_id, max_hasil=50):
    """
    Ambil komentar terbaru dari video YouTube
    """
    try:
        request = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=max_hasil,
            order="time",
            textFormat="plainText"
        )
        response = request.execute()

        hasil = []
        for item in response.get('items', []):
            snippet = item['snippet']['topLevelComment']['snippet']
            hasil.append({
                'id': item['id'],
                'teks': snippet['textDisplay'],
                'penulis': snippet['authorDisplayName'],
                'waktu': snippet['publishedAt'],
            })

        print(f"✅ Berhasil ambil {len(hasil)} komentar")
        return hasil

    except Exception as e:
        print(f"❌ Error ambil komentar: {e}")
        return []

def hapus_komentar(youtube, comment_id):
    """
    Hapus satu komentar berdasarkan ID
    Hanya bisa hapus komentar di channel sendiri!
    """
    try:
        youtube.comments().delete(id=comment_id).execute()
        return True
    except Exception as e:
        print(f"❌ Error hapus: {e}")
        return False

print("✅ Fungsi ambil & hapus komentar siap!")

✅ Fungsi ambil & hapus komentar siap!


In [15]:
import time
from datetime import datetime

def mulai_monitoring(youtube, video_id, interval_menit=1):
    """
    Monitoring komentar otomatis terus-menerus

    youtube        = koneksi YouTube yang sudah login
    video_id       = ID video yang mau dimonitor
    interval_menit = seberapa sering cek komentar baru
    """

    sudah_dicek = set()
    total_dihapus = 0
    total_dicek = 0

    print("=" * 55)
    print("🚀 SISTEM MODERASI JUDOL AKTIF")
    print("=" * 55)
    print(f"📹 Video ID  : {video_id}")
    print(f"⏱️  Interval  : setiap {interval_menit} menit")
    print(f"🕐 Mulai     : {datetime.now().strftime('%H:%M:%S')}")
    print(f"\n⛔ Tekan tombol STOP (■) di Colab untuk berhenti\n")
    print("-" * 55)

    while True:
        try:
            waktu_sekarang = datetime.now().strftime('%H:%M:%S')

            # Ambil komentar terbaru
            komentar_list = ambil_komentar_terbaru(
                youtube, video_id, max_hasil=50
            )

            # Filter hanya yang belum dicek
            komentar_baru = [
                k for k in komentar_list
                if k['id'] not in sudah_dicek
            ]

            if komentar_baru:
                print(f"\n[{waktu_sekarang}] 📥 {len(komentar_baru)} komentar baru!")
                print("-" * 55)

                for komentar in komentar_baru:
                    sudah_dicek.add(komentar['id'])
                    total_dicek += 1

                    # Cek apakah judi
                    judi, alasan = adalah_judi(komentar['teks'])

                    if judi:
                        # Hapus komentar judi
                        berhasil = hapus_komentar(
                            youtube, komentar['id']
                        )
                        if berhasil:
                            total_dihapus += 1
                            print(f"  🗑️  HAPUS!")
                            print(f"      Alasan   : {alasan}")
                            print(f"      Komentar : {komentar['teks'][:70]}")
                            print(f"      Dari     : {komentar['penulis']}")
                            print()
                    else:
                        print(f"  ✅ AMAN")
                        print(f"      Alasan   : {alasan}")
                        print(f"      Komentar : {komentar['teks'][:70]}")
                        print(f"      Dari     : {komentar['penulis']}")
                        print()

                print("-" * 55)
                print(f"📊 Total dicek: {total_dicek} | "
                      f"Total dihapus: {total_dihapus}")

            else:
                print(f"[{waktu_sekarang}] 👀 Memantau... "
                      f"(Dicek: {total_dicek} | "
                      f"Dihapus: {total_dihapus})")

            # Tunggu sebelum cek lagi
            time.sleep(interval_menit * 60)

        except KeyboardInterrupt:
            print("\n\n⛔ Monitoring dihentikan!")
            print("=" * 55)
            print(f"📊 RINGKASAN AKHIR:")
            print(f"   Total komentar dicek  : {total_dicek}")
            print(f"   Total komentar dihapus: {total_dihapus}")
            print(f"   Selesai pukul         : "
                  f"{datetime.now().strftime('%H:%M:%S')}")
            print("=" * 55)
            break

        except Exception as e:
            print(f"\n❌ Error tidak terduga: {e}")
            print("   Mencoba lagi dalam 1 menit...")
            time.sleep(60)

print("✅ Sistem monitoring siap!")

✅ Sistem monitoring siap!


In [ ]:
# ============================================
# GANTI VIDEO_ID dengan ID video kamu!
# Cara dapat: youtube.com/watch?v=XXXXX
#                                  ↑
#                              copy bagian ini
# ============================================

VIDEO_ID = "uwEjIMJXWeQ"

# Validasi dulu sebelum jalan
if VIDEO_ID == "GANTI_DENGAN_VIDEO_ID_KAMU":
    print("⚠️  Belum isi VIDEO_ID!")
    print("    Ganti dulu dengan ID video YouTube kamu")
else:
    print(f"✅ Video ID : {VIDEO_ID}")
    print(f"🔗 Link     : https://youtube.com/watch?v={VIDEO_ID}")
    print("\nMemulai sistem dalam 3 detik...")
    time.sleep(3)

    # Jalankan monitoring!
    mulai_monitoring(youtube, VIDEO_ID, interval_menit=1)

✅ Video ID : uwEjIMJXWeQ
🔗 Link     : https://youtube.com/watch?v=uwEjIMJXWeQ

Memulai sistem dalam 3 detik...
🚀 SISTEM MODERASI JUDOL AKTIF
📹 Video ID  : uwEjIMJXWeQ
⏱️  Interval  : setiap 1 menit
🕐 Mulai     : 14:23:03

⛔ Tekan tombol STOP (■) di Colab untuk berhenti

-------------------------------------------------------
✅ Berhasil ambil 3 komentar

[14:23:03] 📥 3 komentar baru!
-------------------------------------------------------
  🗑️  HAPUS!
      Alasan   : keyword pasti: 'togel'
      Komentar : T0g3l
      Dari     : @midzyy1345

  ✅ AMAN
      Alasan   : normal (0.0% judi)
      Komentar : haloo
      Dari     : @midzyy1345

  ✅ AMAN
      Alasan   : normal (7.0% judi)
      Komentar : bagus bang
      Dari     : @midzyy1345

-------------------------------------------------------
📊 Total dicek: 3 | Total dihapus: 1
✅ Berhasil ambil 2 komentar
[14:24:04] 👀 Memantau... (Dicek: 3 | Dihapus: 1)
✅ Berhasil ambil 2 komentar
[14:25:04] 👀 Memantau... (Dicek: 3 | Dihapus: 1)
✅ Ber